# VWAPSb backtest with local MLflow

Compare explicit model grids, record coverage and errors, and inspect every candidate's saved curves. No winner is chosen automatically and no production settings are changed. MLflow runs locally; results remain in its SQLite database and artifact directory after stopping the service.

Install the notebook and experiment extras and select this project's Python environment. Start with the synthetic demo. Full refill curves retain observed prices; accuracy is measured separately by hiding one physical period at a time.


In [ ]:
PROJECT_ROOT = None  # Example: r"C:\Projects\VWAPSb".
CONFIG_PATH = None  # Explicit config overrides USE_DEMO.
USE_DEMO = True
START_DATE = "2026-09-21"  # Inclusive; None uses first own observation.
END_DATE = "2026-10-09"  # Inclusive; None uses last own observation.
EEX_OFFSET_DAYS = None  # Keep config; use 0 or -1 for separate availability scenarios.
MASK_FRACTION = 1.0  # Fraction examined one case at a time, never simultaneously.
SEED = 42
MLFLOW_PORT = 5055  # Separate from the original project's usual port 5000.
MLFLOW_STORAGE = "output/mlflow"
EXPERIMENT_NAME = "vwapsb-refill"
PARAMETER_GRID = {
    "local_max_steps": [1, 2],
    "history_enabled": [False, True],
}
# Optional external grid:
GRID_PATH = None  # Example: "notebooks/grid.example.json".


In [ ]:
from pathlib import Path
from dataclasses import replace
from datetime import date
import json
import sys
import pandas as pd
from IPython.display import display, Markdown

if PROJECT_ROOT is None:
    current = Path.cwd().resolve()
    ROOT = next((p for p in [current, *current.parents] if (p / "src/vwapsb").is_dir()), None)
    if ROOT is None:
        raise ValueError("Set PROJECT_ROOT to the VWAPSb repository directory.")
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

def project_path(value):
    """Resolve notebook paths relative to the project directory."""
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()

from vwapsb.config import load_config
from vwapsb.demo import create_demo
from vwapsb.io_vwap import load_vwaps
from vwapsb.io_eex import load_eex_books
from vwapsb.mapping import load_mapping
from vwapsb.backtest import configured_candidates
from vwapsb.mlflow_support import start_mlflow_server, run_tracked_backtest, create_experiment_viewer

if CONFIG_PATH is not None:
    config_path = project_path(CONFIG_PATH)
elif USE_DEMO:
    config_path = create_demo(ROOT)
else:
    config_path = ROOT / "config.local.toml" if (ROOT / "config.local.toml").is_file() else ROOT / "config.toml"
config = load_config(config_path)
if EEX_OFFSET_DAYS is not None:
    config = replace(config, eex_offset_days=EEX_OFFSET_DAYS)
if GRID_PATH is not None:
    PARAMETER_GRID = json.loads(project_path(GRID_PATH).read_text(encoding="utf-8-sig"))
mappings = load_mapping(config)
observations = load_vwaps(config.vwap_input, config)
books = load_eex_books(config, mappings)
start = observations.date.min() if START_DATE is None else date.fromisoformat(START_DATE)
end = observations.date.max() if END_DATE is None else date.fromisoformat(END_DATE)
print(f"Config: {config_path}; dates: {start} through {end}")
print(f"EEX offset: {config.eex_offset_days}; fixed production mode: {config.configuration_mode}")
display(pd.DataFrame(configured_candidates(config, mappings, PARAMETER_GRID)))


## Start the local service

The helper starts its own hidden process on 127.0.0.1. It can reuse a service created by this notebook kernel, but never adopts or kills an unrelated service on the same port. Choose another port if it is occupied. The final cell stops the owned process tree and preserves the database and artifacts.


In [ ]:
storage_dir = project_path(MLFLOW_STORAGE)
server = start_mlflow_server(storage_dir, port=MLFLOW_PORT, timeout=45)
display(Markdown(f"**Local MLflow:** [{server.url}]({server.url})"))
print(f"Server log: {server.log_path}")


## Run the experiment

Every candidate examines the same hidden physical contracts. The held-out actual consolidates duplicate prices by positive volume, or by simple mean if none has positive volume. All aliases are removed together. Candidate filters affect usable anchors, not the examination sample. Earlier own history remains available; future own observations are excluded and EEX respects the configured cutoff.

The experiment stores price-level CSV artifacts locally. Summary metrics remain separated by unit; per-curve reports also separate contract kind. There is no automatic calibration/holdout split: use earlier dates to explore, then evaluate the chosen fixed settings on later untouched dates.


In [ ]:
result = run_tracked_backtest(
    observations, books, mappings, config, start, end,
    grid=PARAMETER_GRID, mask_fraction=MASK_FRACTION, seed=SEED,
    tracking_uri=server.url, experiment_name=EXPERIMENT_NAME,
    output_dir=storage_dir / "reports",
)
display(Markdown(f"**Experiment run:** [Open MLflow run]({result.run_url})"))
print(f"Local reports: {result.output_dir}")
display(result.reports["summary"])
display(result.reports["by_curve"])


Coverage counts every selected case, including missing predictions. MAE, RMSE and bias use only finite predictions; lower error with lower coverage is not automatically better. Compare the same cases where possible and inspect difficult products and contract families separately. MLflow displays these measurements; it does not make the evaluation objective valid by itself.


## Inspect a saved candidate

Choose a saved run and either **complete refill** or **withheld predictions**. Complete curves include originals, all configured targets and recognized own contracts; their appearance is not an accuracy measurement. Withheld plots compare predictions with hidden actual prices. Both views allow a date or paired range means and all contract types together.

The viewer loads stored artifacts, without re-running the model or reading current input files. To revisit old experiments in a fresh kernel, run settings, setup and service, then the viewer cell; the experiment cell is unnecessary. Setup currently loads the configured/demo inputs; the saved artifact loader itself does not require them.


In [ ]:
experiment_viewer = create_experiment_viewer(
    server.url, experiment_name=EXPERIMENT_NAME, cache_dir=storage_dir / "viewer_cache",
)
display(experiment_viewer)


## Optional shutdown

Set `STOP_SERVER=True` and run this cell when finished. Stored runs remain available the next time you start the service with the same storage directory.


In [ ]:
STOP_SERVER = False
if STOP_SERVER:
    server.stop()
    print("Local MLflow service stopped; stored experiments were preserved.")
